# Adicional: regresión múltiple, fundamentos y supuestos (práctica)

**Tópicos de Finanzas Avanzadas, UPAO 2026-2. Material adicional de consulta, no evaluado.**

Tema del CFA Level II (Quantitative Methods, *Basics of Multiple Regression and Underlying Assumptions*) que no forma parte del sílabo del curso, pero que complementa las semanas 2 y 9 (estimación del beta por MCO, modelos de factores). Modelo de referencia (retornos mensuales):

$$r_{IGBVL,t} = b_0 + b_1\, r_{cobre,t} + b_2\, \Delta TC_t + \varepsilon_t$$

Hipótesis económicas: $b_1 > 0$ (el Perú es exportador de cobre) y $b_2 < 0$ (la depreciación del sol suele coincidir con salida de capitales).

> Ejecuta cada celda con `Shift+Enter`. Si es tu primera vez, completa antes la [guía de instalación](../../00_setup/guia_instalacion.md).


## 0. Verificación del entorno

In [ ]:
# Preparación del entorno (funciona en VS Code y en Google Colab)
import sys, os

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    # En Colab solo se abre este notebook: clonamos el repo para tener utils/ y data/
    if not os.path.exists("/content/repo"):
        !git clone -q https://github.com/JonathanRosasV/topicos-finanzas-upao.git /content/repo
    %cd /content/repo/adicionales/regresion_multiple
    !pip install -q yfinance

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf

print("Python:", sys.version.split()[0], "| Colab:", EN_COLAB, "| Entorno OK")


## 1. Descarga de datos por API

**Regla del curso:** nada de copiar datos a mano, todo por API, para que el notebook sea reproducible.

- Cobre: futuros `HG=F` desde Yahoo Finance.
- Tipo de cambio e índice bursátil: BCRPData (`utils/bcrp.py`).

> Los códigos de serie del BCRP se buscan en el portal [BCRPData](https://estadisticas.bcrp.gob.pe/estadisticas/series/). Verifica en clase que sigan vigentes.

In [ ]:
sys.path.append("../..")   # raíz del repo (utils/); en Colab ya estamos dentro del repo clonado
from utils.bcrp import get_series

INICIO, FIN = "2015-1", "2026-6"

# Tipo de cambio interbancario promedio (mensual): verificar código en BCRPData
tc = get_series("PN01207PM", INICIO, FIN)

# Índice General BVL (mensual): verificar código en BCRPData
igbvl = get_series("PN01712BM", INICIO, FIN)

tc.tail()

In [ ]:
# Cobre: futuros HG=F, cierre mensual
cobre = yf.download("HG=F", start="2015-01-01", interval="1mo", auto_adjust=True)["Close"]
cobre = cobre.squeeze()
cobre.tail()

## 2. Construcción de la base: retornos mensuales

In [ ]:
df = pd.DataFrame({
    "igbvl": igbvl.resample("MS").last(),
    "cobre": cobre.resample("MS").last(),
    "tc":    tc.resample("MS").last(),
}).dropna()

ret = df.pct_change().dropna()
ret.columns = ["r_igbvl", "r_cobre", "d_tc"]

print(ret.describe().round(4))
ret.plot(subplots=True, figsize=(9, 6), title="Retornos mensuales");

## 3. Estimación por MCO con `statsmodels`

In [ ]:
X = sm.add_constant(ret[["r_cobre", "d_tc"]])
modelo = sm.OLS(ret["r_igbvl"], X, missing="drop").fit()
print(modelo.summary())

### Lectura CFA de la salida

Identifica en la tabla: coeficientes, errores estándar, estadísticos $t$, $R^2$ y $R^2$ ajustado, estadístico $F$.

**Interpretación (estilo examen):** $\hat b_1$ = cambio esperado en el retorno del IGBVL ante un cambio de 1 punto porcentual en el retorno del cobre, **manteniendo constante** la variación del tipo de cambio (*holding all other independent variables constant*).

Escribe aquí tu interpretación de $\hat b_1$ y $\hat b_2$ con los números obtenidos:

- $\hat b_1$: ...
- $\hat b_2$: ...

## 4. Primer diagnóstico visual: residuos vs. valores ajustados

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(modelo.fittedvalues, modelo.resid, alpha=0.7)
ax.axhline(0, color="gray", ls="--")
ax.set_xlabel("Valores ajustados $\\hat{Y}$")
ax.set_ylabel("Residuos $e$")
ax.set_title("¿Homocedasticidad? (la semana 2 lo probamos formalmente)")
plt.show()

## 5. Cierre

- Este material es de consulta y no tiene entrega. Si quieres seguir, `ejercicios.ipynb` propone replicar el análisis con otra acción minera y cuatro ítems tipo CFA.
- Los diagnósticos formales del modelo (heterocedasticidad, autocorrelación, multicolinealidad) y las series de tiempo están descritos en el índice de `adicionales/README.md` como temas de lectura.
